# Extractive Text Summarization berbasis TF-IDF

Studi kasus: Jurnal **Perkembangan dan Peran OPAC pada Aplikasi CIP (Cerah Informasi Pustaka) untuk Temu Kembali Informasi di Perpustakaan Universitas Tridinanti Palembang** (Betari Ayu Elsadantia, 2023).

## Aturan implementasi (dari dosen)
1. **Input adalah 1 dokumen utuh** (raw text), bukan D1/D2/D3 manual.
2. **RegEx memotong bab**: hanya isi `PENDAHULUAN` s/d akhir `HASIL PENELITIAN DAN PEMBAHASAN`. Abstrak, Kesimpulan, Daftar Pustaka harus terbuang otomatis.
3. **Paragraf = sub-dokumen**: deteksi paragraf via karakter enter (`\n\n`). Paragraf 1 = D1, paragraf 2 = D2, dst.
4. **Scoring & summarization**: hitung TF-IDF seluruh paragraf (manual TF, DF, IDF), lakukan ranking dengan TF-IDF & VSM, lalu ekstrak kalimat terbaik tiap paragraf.

## Pipeline notebook
1. **Ekstraksi PDF → raw text** (pdfplumber)
2. **RegEx** memotong bab (PENDAHULUAN s/d HASIL PENELITIAN DAN PEMBAHASAN)
3. **Split paragraf** (`\n\n`) → daftar sub-dokumen D1, D2, …
4. **Preprocessing** (case folding → cleaning → tokenizing → stopword removal → stemming)
5. **TF-IDF Manual** (vocab, TF, DF, IDF, TF-IDF)
6. **Bobot TF-IDF per Paragraf** (Top-5 keyword tiap paragraf)
7. **Ranking dengan TF-IDF** (query-based scoring)
8. **Ranking dengan VSM** (cosine similarity manual)
9. **Perbandingan ranking** TF-IDF vs VSM
10. **Summarization** (kalimat skor TF-IDF tertinggi tiap paragraf)

## 0. Install dependencies (sekali saja)

In [ ]:
# =====================================================================
# CELL 1 — INSTALL DEPENDENCIES (sekali jalan)
# =====================================================================
# Tujuan : memastikan semua library yang dipakai pipeline ini tersedia.
# Library:
#   - pdfplumber  -> ekstrak teks dari PDF lengkap dengan posisi (x0, fontname)
#                    yang nantinya dipakai untuk mendeteksi indent paragraf
#                    dan sub-judul bold pada Langkah 1.
#   - PySastrawi  -> stopword remover + stemmer Bahasa Indonesia
#                    (algoritma Nazief-Adriani), dipakai pada preprocessing.
#   - numpy       -> operasi vektor/dot-product untuk cosine similarity (VSM).
#   - pandas      -> menampilkan tabel dokumen, vocab, ranking, summary.
#
# Catatan environment:
#   - Di Google Colab, `!pip install` aman dijalankan langsung.
#   - Di Linux modern (Debian/Ubuntu) seringkali muncul error
#     "externally-managed-environment" (PEP 668). Solusinya: buat virtual env
#     dulu (`python -m venv .venv && source .venv/bin/activate`) lalu pip install
#     di dalam venv tersebut, ATAU jalankan notebook ini di Google Colab.
# =====================================================================

# Pasang library yang dibutuhkan (jalankan sekali di environment baru / Colab)
!pip install -q pdfplumber PySastrawi numpy pandas


In [ ]:
# =====================================================================
# CELL 2 — IMPORT LIBRARY & KONFIGURASI GLOBAL
# =====================================================================
# Cell ini melakukan tiga hal:
#
#   1) IMPORT MODUL
#      - math       : math.log10 untuk rumus IDF = log10(N / df).
#      - re         : seluruh operasi RegEx (potong bab, splitter kalimat,
#                     pelindung singkatan, dll.).
#      - Counter    : menghitung frekuensi token (TF) dan frekuensi term query.
#      - numpy      : np.dot dan np.linalg.norm untuk cosine similarity.
#      - pandas     : DataFrame untuk tampilan tabel dokumen, ranking, summary.
#      - pdfplumber : ekstrak baris teks PDF beserta metadata posisi & font.
#      - Sastrawi   : StopWordRemoverFactory (buang kata umum Indonesia)
#                     + StemmerFactory (stemming Nazief-Adriani).
#
#   2) KONFIGURASI TAMPILAN PANDAS
#      - max_rows / max_columns 200 : agar tabel besar tidak terpotong "...".
#      - max_colwidth = None         : kolom teks panjang (paragraf/kalimat
#                                       summary) ditampilkan utuh.
#      - expand_frame_repr = False   : DataFrame tidak dipecah jadi banyak
#                                       baris saat lebar terminal sempit.
#
#   3) INISIASI OBJEK SASTRAWI (sekali saja, mahal kalau dibikin per call)
#      - stopword_remover : objek yang punya method .remove(teks).
#      - stemmer          : objek yang punya method .stem(token).
# =====================================================================

import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import pdfplumber

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print('Import dan konfigurasi selesai. Teks tabel tidak dipotong.')


## LANGKAH 1 — Ekstraksi `Jurnal.pdf` menjadi raw text

Tantangan: pdfplumber memecah teks per baris (line-wrap), bukan per paragraf. Untuk
menghormati aturan dosen ("paragraf dipisah karakter enter"), kita memanfaatkan posisi
horizontal `x0` tiap baris:

* Baris **rata kiri** (`x0 ≈ 104.9`) → lanjutan paragraf yang sama.
* Baris **terindentasi** (`x0 ≈ 133.3`) → awal paragraf baru → sisipkan `\n\n`.
* Header bab (PENDAHULUAN, METODE, dll.) berdiri sendiri di kolom kiri → diberi `\n\n` sendiri.

Hasilnya satu string raw text yang strukturnya "berantakan tapi paragraf jelas".

In [ ]:
# =====================================================================
# CELL 3 — EKSTRAKSI PDF MENJADI RAW TEXT (LANGKAH 1)
# =====================================================================
# MASALAH:
#   pdfplumber memecah teks per-baris (line-wrap), bukan per-paragraf.
#   Padahal aturan dosen: "paragraf dipisah dengan karakter enter" -> kita
#   harus menyusun ulang baris-baris ini menjadi paragraf, lalu memisahkannya
#   dengan '\n\n' supaya Langkah 3 bisa split('\n\n').
#
# STRATEGI DETEKSI BATAS PARAGRAF:
#   pdfplumber memberi tiap baris dua metadata penting:
#     - ln['x0']     -> posisi horizontal awal baris (px).
#     - ln['chars']  -> list karakter; chars[0]['fontname'] = font huruf
#                       pertama -> bisa dipakai cek bold.
#
#   Aturan keputusan per baris:
#     (a) Kalau x0 > BATAS_INDENT (= 125)  -> baris terindentasi
#                                             -> AWAL paragraf baru -> flush()
#                                             dan mulai buffer baru.
#     (b) Kalau cocok pola UPPERCASE pendek (PENDAHULUAN, METODE, dll.)
#         -> header bab -> jadikan paragraf sendiri.
#     (c) Kalau font baris = 'Arial-BoldMT' DAN kita berada di "boundary"
#         (baris sebelumnya berakhir titik/!/?), berarti baris bold ini =
#         SUB-JUDUL Title Case (misal "OPAC", "Sistem Temu Kembali Informasi").
#         Sub-judul TIDAK termasuk isi paragraf -> dibuang sepenuhnya.
#         Catatan: pembedanya dengan kutipan judul di tengah teks adalah
#         posisi boundary. Kutipan judul muncul di tengah kalimat, jadi
#         prev_end_with_period = False di posisi itu.
#     (d) Selain itu (rata kiri, bukan bold, bukan header) -> baris ini
#         adalah lanjutan paragraf sebelumnya -> append ke buffer dengan
#         spasi.
#
# STATE MACHINE:
#   - prev_end_with_period: True kalau baris sebelumnya berakhir titik/!/?
#                            -> menandakan kita di "boundary" antar-paragraf.
#                            -> awal dokumen dianggap True (boundary).
#   - skip_streak         : True kalau baris bold sebelumnya juga sub-judul
#                            -> handle sub-judul yang multi-baris (mis.
#                            "Perkembangan dan Peran" + "OPAC").
#
# FILTER PAGE-FOOTER & RUNNING-HEADER:
#   pola_sampah membuang baris-baris seperti:
#     - "10 Jurnal Multidisipliner..."        (footer halaman)
#     - "Perpustakaan Universitas Tridinanti" (running header)
#     - "Betari Ayu", "Elsadantia"            (running header nama penulis)
#
# OUTPUT:
#   - teks_jurnal: 1 string raw text (~22.811 char) berisi seluruh konten
#     PDF tersusun ulang sebagai paragraf, dipisah '\n\n'.
#     (Abstrak masih ada di sini -> akan dibuang Langkah 2.)
# =====================================================================

PATH_PDF = 'Jurnal.pdf'

def ekstrak_pdf_jadi_raw_text(path_pdf: str) -> str:
    """Ekstrak PDF jurnal jadi 1 string raw text dengan paragraf dipisah \\n\\n.

    Strategi:
      - Posisi x0 untuk mendeteksi indent (= awal paragraf baru).
      - Pola UPPERCASE pendek untuk header bab (PENDAHULUAN/METODE/dll.).
      - Font 'Arial-BoldMT' untuk sub-judul Title Case (Perpustakaan, OPAC,
        Sistem Temu Kembali Informasi, dll.) yang dibuang sepenuhnya.
        Pembedanya dengan kutipan judul: sub-judul muncul SETELAH paragraf
        yang berakhir dengan tanda baca akhir kalimat (./?/!).
      - Page-footer & running-header juga disaring.
    """
    BATAS_INDENT = 125
    bagian_paragraf: list[str] = []
    buffer = ''

    def flush():
        nonlocal buffer
        if buffer.strip():
            bagian_paragraf.append(buffer.strip())
        buffer = ''

    pola_header_bab = re.compile(r'^[A-Z][A-Z\s]{3,40}$')
    pola_sampah = re.compile(
        r'^(?:\d+\s+Jurnal Multidisipliner|'
        r'Perkembangan Dan Peran Opac Pada Aplikasi|'
        r'Informasi Pustaka\) Untuk Temu Kembali|'
        r'Perpustakaan Universitas Tridinanti Palembang\s*$|'
        r'Betari Ayu\s*$|Elsadantia\s*$)',
        flags=re.IGNORECASE,
    )

    # State: apakah baris sebelumnya berakhir dengan tanda baca akhir kalimat?
    # True berarti kita berada di "boundary" antar-paragraf -> baris bold di
    # posisi ini sangat mungkin sub-judul.
    prev_end_with_period = True   # awal dokumen dianggap boundary
    skip_streak = False           # apakah baris bold sebelumnya juga sub-judul

    with pdfplumber.open(path_pdf) as pdf:
        for hal in pdf.pages:
            for ln in hal.extract_text_lines():
                teks_baris = ln['text'].strip()
                if not teks_baris or pola_sampah.search(teks_baris):
                    continue

                chars = ln.get('chars', [])
                font_pertama = chars[0].get('fontname', '') if chars else ''
                line_bold = 'Arial-BoldMT' in font_pertama

                terindentasi = ln['x0'] > BATAS_INDENT
                header_bab = bool(pola_header_bab.match(teks_baris))

                # Sub-judul Title Case: bold + posisi boundary + bukan UPPERCASE bab
                # Atau: bold + lanjutan dari sub-judul multi-baris sebelumnya
                sub_judul_title_case = line_bold and not header_bab and (
                    prev_end_with_period or skip_streak
                )

                if sub_judul_title_case:
                    flush()                 # tutup paragraf sebelumnya
                    skip_streak = True       # next bold line juga = continuation
                    # state prev_end_with_period dipertahankan (masih di boundary)
                    continue

                # Update state untuk baris berikutnya
                skip_streak = False
                prev_end_with_period = bool(re.search(r'[.!?]\s*$', teks_baris))

                if terindentasi or header_bab:
                    flush()
                    buffer = teks_baris
                    if header_bab:
                        flush()
                else:
                    buffer = (buffer + ' ' + teks_baris).strip()
    flush()

    return '\n\n'.join(bagian_paragraf)


teks_jurnal = ekstrak_pdf_jadi_raw_text(PATH_PDF)
print(f'Total karakter raw text: {len(teks_jurnal):,}')
print('=' * 78)
print('CUPLIKAN 800 KARAKTER PERTAMA:')
print('=' * 78)
print(teks_jurnal[:800])


## LANGKAH 2 — RegEx memotong bab

Hanya isi mulai dari `PENDAHULUAN` sampai sebelum `KESIMPULAN` yang akan kita pakai.
Bagian Abstrak (di atas PENDAHULUAN) dan Kesimpulan + Daftar Pustaka (setelah Hasil)
akan otomatis terbuang.

In [ ]:
# =====================================================================
# CELL 4 — REGEX MEMOTONG BAB PENDAHULUAN..KESIMPULAN (LANGKAH 2)
# =====================================================================
# TUJUAN:
#   Aturan dosen: "hanya isi mulai dari PENDAHULUAN sampai akhir HASIL
#   PENELITIAN DAN PEMBAHASAN yang dipakai. Abstrak, Kesimpulan, Daftar
#   Pustaka harus terbuang otomatis."
#
#   Trik: pakai kata 'KESIMPULAN' sebagai penanda akhir bab Hasil &
#   Pembahasan, sehingga isi DI ANTARA 'PENDAHULUAN' dan 'KESIMPULAN'
#   adalah persis bagian yang kita mau (Pendahuluan + Metode + Hasil).
#
# ANATOMI REGEX:
#   r'PENDAHULUAN\s*\n(.*?)\n\s*KESIMPULAN'
#     - PENDAHULUAN     : penanda awal (literal).
#     - \s*\n           : whitespace + newline setelah header.
#     - (.*?)           : LAZY capture group ke-1 = isi bab yang ingin
#                         kita ambil. Lazy (*?) supaya match BERHENTI di
#                         kemunculan KESIMPULAN PERTAMA, bukan terakhir
#                         (kalau pakai .* serakah, dia akan menjangkau
#                         sampai 'KESIMPULAN' lain di Daftar Pustaka).
#     - \n\s*KESIMPULAN : penanda akhir.
#
# FLAG REGEX:
#   - re.DOTALL    : titik (.) ikut cocok dengan '\n', supaya capture
#                    bisa lintas paragraf (default-nya tidak).
#   - re.IGNORECASE: jaga-jaga kalau kapitalisasi sedikit bervariasi.
#
# CARA AMBIL HASIL:
#   - search()      mengeksekusi pencarian.
#   - match.group(1) -> isi capture group ke-1 (apa yang ada di antara
#                       penanda awal & akhir).
#
# OUTPUT:
#   - isi_bab : ~15.844 karakter (turun dari 22.811). Selisih karakter
#               itulah Abstrak + Kesimpulan + Daftar Pustaka yang sudah
#               terbuang otomatis.
# =====================================================================

# ---- REGEX UTAMA: tangkap teks mulai dari kata kunci 'PENDAHULUAN'
# sampai sebelum kata kunci 'KESIMPULAN' (penanda akhir bab Hasil & Pembahasan).
# Flag re.DOTALL membuat titik (.) ikut cocok dengan karakter newline,
# sehingga grup (.*?) bisa melahap lintas paragraf. Penggunaan *? (lazy)
# memastikan match berhenti di kemunculan KESIMPULAN yang PERTAMA.
pola_bab = re.compile(
    r'PENDAHULUAN\s*\n(.*?)\n\s*KESIMPULAN',
    flags=re.DOTALL | re.IGNORECASE,
)

match_bab = pola_bab.search(teks_jurnal)   # eksekusi pencarian RegEx pada teks mentah
if not match_bab:
    raise ValueError('Bab PENDAHULUAN..KESIMPULAN tidak ditemukan.')

isi_bab = match_bab.group(1).strip()        # group(1) = isi di antara dua penanda

print(f'Karakter sebelum RegEx: {len(teks_jurnal):,}')
print(f'Karakter setelah RegEx: {len(isi_bab):,}  (Abstrak + Kesimpulan + Pustaka terbuang)')
print('=' * 78)
print('CUPLIKAN 400 KARAKTER PERTAMA HASIL POTONG:')
print('=' * 78)
print(isi_bab[:400], '...')
print('=' * 78)
print('CUPLIKAN 400 KARAKTER TERAKHIR HASIL POTONG:')
print('=' * 78)
print('...', isi_bab[-400:])


## LANGKAH 3 — Pecah jadi paragraf (sub-dokumen)

Setiap paragraf dipisah `\n\n`, jadi cukup `.split('\n\n')`. Sub-judul pendek
(mis. `OPAC`, `Perkembangan OPAC`) yang tidak memuat kalimat valid kita buang dengan
ambang panjang minimum 30 karakter.

In [ ]:
# =====================================================================
# CELL 5 — PECAH JADI PARAGRAF / SUB-DOKUMEN D1..Dn (LANGKAH 3)
# =====================================================================
# IDE:
#   Karena Cell 3 sudah memastikan "antar paragraf dipisah '\n\n' ",
#   memecah dokumen menjadi sub-dokumen tinggal split('\n\n').
#   Tiap elemen hasil split = satu paragraf = satu "dokumen" Di dalam
#   model TF-IDF/VSM kita.
#
# DUA FILTER NOISE:
#   1) len(p) >= 30
#      -> buang paragraf super pendek seperti "OPAC" atau judul tabel
#         singkat. Threshold 30 dipilih karena rata-rata sub-judul
#         pendek <= 30 karakter sedangkan kalimat valid biasanya > 50.
#   2) bukan UPPERCASE-only (regex r'^[A-Z\s]+$')
#      -> buang sisa header bab yang lolos seperti
#         "HASIL PENELITIAN DAN PEMBAHASAN".
#
# STRUKTUR DATA HASIL:
#   - dokumen        : dict {'D1': '<teks paragraf 1>', 'D2': ..., ...}
#                      Notasi 'Di' diadopsi dari notebook STKI_TFIDF_VSM
#                      sebelumnya supaya konsisten.
#   - label_dokumen  : dict {'D1': 'Paragraf 1', ...} -> dipakai untuk
#                      kolom "Bagian Jurnal" pada tabel ringkasan.
#   - df_korpus      : DataFrame berisi (Dokumen, Label, Teks Asli)
#                      ditampilkan supaya kita bisa cross-check apakah
#                      pemecahan paragraf masuk akal.
#
# OUTPUT:
#   26 paragraf (D1..D26) untuk jurnal ini. Angka ini akan jadi N pada
#   rumus IDF di Cell 7.
# =====================================================================

# ---- PEMISAHAN PARAGRAF: gunakan split('\n\n') sesuai instruksi dosen.
# Setiap elemen hasil split kita anggap sebagai 1 sub-dokumen (D1, D2, ...).
paragraf_kasar = isi_bab.split('\n\n')

# Filter:
#  - Buang paragraf < 30 karakter (sub-judul pendek seperti "OPAC")
#  - Buang baris UPPERCASE-only (sisa header bab seperti "HASIL PENELITIAN DAN PEMBAHASAN")
pola_uppercase_only = re.compile(r'^[A-Z\s]+$')

daftar_paragraf = [
    p.strip() for p in paragraf_kasar
    if len(p.strip()) >= 30 and not pola_uppercase_only.match(p.strip())
]

# Bangun dict dokumen & label sesuai pola notebook STKI_TFIDF_VSM_Final
dokumen = {f'D{i+1}': p for i, p in enumerate(daftar_paragraf)}
label_dokumen = {f'D{i+1}': f'Paragraf {i+1}' for i in range(len(daftar_paragraf))}

print(f'Total paragraf terdeteksi (sub-dokumen): {len(daftar_paragraf)}')
print('=' * 78)
df_korpus = pd.DataFrame({
    'Dokumen': list(dokumen.keys()),
    'Bagian Jurnal': [label_dokumen[d] for d in dokumen.keys()],
    'Teks Asli': list(dokumen.values()),
})
display(df_korpus)


## Preprocessing

Pipeline `preprocess(text)`:
1. Case folding
2. Cleaning (regex: hapus angka dan tanda baca)
3. Tokenizing
4. Stopword Removal (`StopWordRemoverFactory`)
5. Stemming (`StemmerFactory`)

In [ ]:
# =====================================================================
# CELL 6 — PREPROCESSING TEKS (LANGKAH 4)
# =====================================================================
# Pipeline 5 tahap standar IR untuk Bahasa Indonesia:
#
#   1) CASE FOLDING   : text.lower()
#      -> "Perpustakaan" dan "perpustakaan" jadi token yang sama.
#
#   2) CLEANING       : re.sub(r'[^a-z\s]', ' ', text)
#      -> Buang ANGKA dan TANDA BACA. Pola [^a-z\s] mengganti apapun
#         yang BUKAN huruf kecil atau whitespace dengan spasi.
#         Catatan: karena dijalankan SETELAH lower(), kita cukup pakai
#         a-z (tidak perlu A-Z).
#
#   3) TOKENIZING     : text.split()
#      -> Pecah jadi list token berdasarkan whitespace.
#
#   4) STOPWORD REMOVAL (Sastrawi)
#      -> Buang kata umum Bahasa Indonesia yang tidak informatif:
#         "yang", "dan", "di", "ke", "dari", "ini", "itu", dll.
#         Sastrawi bekerja di level string (bukan list), makanya kita
#         join() dulu jadi string, panggil .remove(), lalu split() lagi.
#
#   5) STEMMING (Sastrawi - Nazief & Adriani)
#      -> Reduksi ke kata dasar:
#         "perpustakaan" -> "pustaka"
#         "menggunakan" -> "guna"
#         "informasi" -> "informasi" (kata dasar tidak berubah).
#         Penting supaya "pustaka" di D1 dan "perpustakaan" di D2
#         dianggap term yang SAMA pada perhitungan TF-IDF.
#
# RETURN: list[str] = list token bersih siap pakai.
#
# OUTPUT GLOBAL:
#   - hasil_preproses: dict {'D1': ['dunia','gantung',...], 'D2': [...], ...}
#     Inilah corpus yang masuk ke TF-IDF.
# =====================================================================

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    teks_tanpa_stopword = stopword_remover.remove(' '.join(tokens))
    tokens_tanpa_stopword = teks_tanpa_stopword.split()
    tokens_stem = [stemmer.stem(token) for token in tokens_tanpa_stopword]
    return tokens_stem

hasil_preproses = {nama_dok: preprocess(teks) for nama_dok, teks in dokumen.items()}

df_preproses = pd.DataFrame({
    'Dokumen': list(hasil_preproses.keys()),
    'Bagian Jurnal': [label_dokumen[d] for d in hasil_preproses.keys()],
    'Token Hasil Preprocessing': [' '.join(v) for v in hasil_preproses.values()]
})
display(df_preproses)


## Perhitungan TF-IDF Manual

In [ ]:
# =====================================================================
# CELL 7 — PERHITUNGAN TF-IDF MANUAL (LANGKAH 5)
# =====================================================================
# Empat besaran utama dihitung di sini, semuanya secara manual:
#
#   (1) VOCAB
#       vocab = himpunan TERM unik di seluruh dokumen, di-sort.
#       Disort agar urutan kolom matriks deterministik (penting nanti
#       saat membentuk vektor np.array di Cell 9 - VSM).
#
#   (2) TF (Term Frequency) — DINORMALISASI
#       tf[d][t] = jumlah_kemunculan_t_di_d / total_token_di_d
#       Memakai NORMALISASI (bagi total token) supaya paragraf panjang
#       tidak otomatis menang. Total token = max(len, 1) untuk hindari
#       pembagian nol pada paragraf kosong (jaga-jaga).
#
#   (3) DF (Document Frequency)
#       df[t] = jumlah dokumen yang MENGANDUNG term t (bukan jumlah
#       kemunculan). Dipakai memprediksi "seberapa umum" term t di
#       korpus.
#
#   (4) IDF (Inverse Document Frequency)
#       idf[t] = log10(N / df[t])   dengan N = jumlah dokumen.
#       Pakai log10 (basis 10) sesuai konvensi banyak buku IR Indonesia.
#       Sifat penting:
#         - Term yang muncul di SEMUA dokumen -> idf = log10(N/N) = 0
#           -> bobot TF-IDF jadi 0 -> term umum dianggap tidak
#           informatif.
#         - Term langka -> idf besar -> bobot tinggi.
#       Catatan: tidak ada smoothing (tidak +1). Aman karena vocab
#       dibangun dari dokumen-dokumen ini sendiri, jadi df[t] >= 1
#       selalu (no division by zero).
#
#   (5) TF-IDF
#       tfidf[d][t] = tf[d][t] * idf[t]
#
# STRUKTUR DATA:
#   Semua disimpan sebagai dict-bersarang (bukan numpy matrix), supaya
#   indexing pakai key (d, t) lebih intuitif untuk debugging. Konversi
#   ke np.array baru dilakukan di Cell 9 saat butuh dot product VSM.
#
# UNTUK JURNAL INI: vocab = 393 term, N = 26 dokumen (paragraf).
# =====================================================================

nama_dokumen = list(hasil_preproses.keys())
N = len(nama_dokumen)

vocab = sorted({term for tokens in hasil_preproses.values() for term in tokens})

# TF manual
tf = {}
for dok, tokens in hasil_preproses.items():
    counter = Counter(tokens)
    total_token = len(tokens) if len(tokens) > 0 else 1
    tf[dok] = {term: counter.get(term, 0) / total_token for term in vocab}

# DF manual
df_freq = {term: sum(1 for dok in nama_dokumen if tf[dok][term] > 0) for term in vocab}

# IDF manual
idf = {term: math.log10(N / df_freq[term]) if df_freq[term] > 0 else 0.0 for term in vocab}

# TF-IDF manual
tfidf = {
    dok: {term: tf[dok][term] * idf[term] for term in vocab}
    for dok in nama_dokumen
}

print(f'Jumlah vocabulary (term unik): {len(vocab)} term')
print(f'Jumlah dokumen: {N}')


## Bobot TF-IDF per Paragraf (Dokumen)

In [ ]:
# =====================================================================
# CELL 8 — TOP-5 KEYWORD UTAMA PER PARAGRAF (LANGKAH 6)
# =====================================================================
# TUJUAN:
#   Menampilkan, untuk tiap paragraf D1..D26, 5 term dengan bobot
#   TF-IDF TERTINGGI. Ini berfungsi sebagai "ringkasan term" yang
#   dapat dipakai untuk mengintip topik dominan tiap paragraf
#   sebelum kita menyentuh ranking maupun summarization.
#
# CARA KERJA:
#   for setiap dokumen D:
#     1) Filter term yang punya bobot > 0 di D
#        (hilangkan term yang tidak muncul / IDF=0).
#     2) Sort menurun berdasarkan skor TF-IDF.
#     3) Ambil 5 term teratas.
#     4) Cetak dengan visualisasi bar ASCII:
#        bar = '█' * int(skor * 600)
#        - faktor 600 dipilih hanya supaya visual bar terlihat;
#          bukan bagian dari rumus, hanya estetika output terminal.
#
# OUTPUT:
#   - Cetak per dokumen: ranking 1-5 + bar visual.
#   - bobot_data: list-of-dict yang diconvert ke df_bobot
#     -> tabel ringkasan kolom [Dokumen, Bagian Jurnal,
#        Keyword Utama (Top-5), Skor TF-IDF Tertinggi].
# =====================================================================

top_k = 5

print('BOBOT TF-IDF PER PARAGRAF (DOKUMEN)')
print('=' * 65)

bobot_data = []

for dok in nama_dokumen:
    skor_term = {t: tfidf[dok][t] for t in vocab if tfidf[dok][t] > 0}
    top_terms = sorted(skor_term.items(), key=lambda x: x[1], reverse=True)[:top_k]

    print(f'\n{dok} — {label_dokumen[dok]}')
    print('-' * 65)
    for rank, (term, skor) in enumerate(top_terms, 1):
        bar = '█' * int(skor * 600)
        print(f'  {rank}. {term:<22} TF-IDF: {skor:.6f}  {bar}')

    keyword_str = ', '.join([t for t, _ in top_terms])
    bobot_data.append({
        'Dokumen': dok,
        'Bagian Jurnal': label_dokumen[dok],
        'Keyword Utama (Top-5)': keyword_str,
        'Skor TF-IDF Tertinggi': round(top_terms[0][1], 6) if top_terms else 0
    })

print('\n')
print('=' * 65)
print('TABEL RINGKASAN KEYWORD UTAMA PER PARAGRAF')
print('=' * 65)
df_bobot = pd.DataFrame(bobot_data)
display(df_bobot)


## Ranking dengan TF-IDF

In [ ]:
# =====================================================================
# CELL 9 — RANKING DOKUMEN DENGAN TF-IDF (LANGKAH 7)
# =====================================================================
# MODEL:
#   "TF-IDF summation" -> skor relevansi sebuah dokumen terhadap query
#   adalah JUMLAH bobot TF-IDF dari term-term query yang ada di
#   dokumen tersebut.
#
#   skor(d, q) = Σ_{t ∈ q} tfidf[d][t]
#
#   Catatan: ini BUKAN cosine similarity. Tidak ada normalisasi
#   panjang vektor, jadi paragraf yang panjang dengan banyak match
#   cenderung diuntungkan. Ini sengaja agar kita bisa membandingkan
#   karakteristiknya dengan VSM cosine pada Cell 10.
#
# QUERY:
#   "perpustakaan opac sistem temu kembali informasi aplikasi cip"
#   Setelah preprocess() (case fold + clean + stopword + stem):
#     -> ['pustaka', 'opac', 'sistem', 'temu', 'informasi',
#         'aplikasi', 'cip']
#   Term query yang TIDAK ada di vocab (mis. typo) di-filter dengan
#   `if term in vocab`, supaya `tfidf[dok][term]` tidak KeyError.
#
# OUTPUT:
#   ranking_tfidf: DataFrame [Dokumen, Bagian Jurnal, Skor TF-IDF,
#                             Ranking TF-IDF] yang sudah diurutkan.
#
# INTERPRETASI:
#   Top-5 untuk jurnal ini biasanya: D4, D25, D18, D26, D20.
#   D6 = 0 -> paragraf yang sama sekali tidak menyebut term query.
# =====================================================================

query = 'perpustakaan opac sistem temu kembali informasi aplikasi cip'
query_tokens = preprocess(query)
query_terms = [term for term in query_tokens if term in vocab]

skor_tfidf = {}
for dok in nama_dokumen:
    skor_tfidf[dok] = sum(tfidf[dok][term] for term in query_terms)

ranking_tfidf = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Bagian Jurnal': [label_dokumen[d] for d in nama_dokumen],
    'Skor TF-IDF': [skor_tfidf[d] for d in nama_dokumen]
}).sort_values('Skor TF-IDF', ascending=False).reset_index(drop=True)
ranking_tfidf['Ranking TF-IDF'] = ranking_tfidf.index + 1

print('Query:', query)
print('Term query setelah preprocessing:', query_terms)
display(ranking_tfidf)


## Ranking dengan VSM (Cosine Similarity)

In [ ]:
# =====================================================================
# CELL 10 — RANKING DOKUMEN DENGAN VSM / COSINE SIMILARITY (LANGKAH 8)
# =====================================================================
# MODEL VSM (Vector Space Model):
#   - Setiap dokumen direpresentasikan sebagai VEKTOR berdimensi |vocab|
#     dengan komponen ke-i = bobot TF-IDF term ke-i di dokumen tersebut.
#   - Query JUGA dijadikan vektor pada ruang vocab yang sama
#     (TF query dihitung relatif panjang query, lalu dikalikan IDF
#     korpus -> "query vector").
#   - Relevansi diukur dengan COSINE SIMILARITY:
#
#         cos(A, B) = (A · B) / (||A|| * ||B||)
#
#   Properti penting:
#     - cos = 1  -> arah vektor sama persis
#     - cos = 0  -> orthogonal (tidak ada term overlap berbobot)
#     - Karena DIBAGI dengan norma vektor, cosine TIDAK bias terhadap
#       panjang dokumen. Ini perbedaan utamanya dengan ranking summation
#       di Cell 9.
#
# IMPLEMENTASI MANUAL (sesuai aturan dosen, tidak pakai sklearn):
#   - cosine_similarity_manual(a, b):
#       pembilang = np.dot(a, b)              -> dot product
#       penyebut  = norm(a) * norm(b)         -> magnitude
#       guard     : kalau penyebut 0 (vektor nol) -> return 0 untuk
#                   menghindari ZeroDivisionError.
#
# URUTAN VEKTOR:
#   Komponen ke-i pada vektor dokumen DAN query dipetakan dari urutan
#   `vocab` yang sama (vocab sudah di-sort di Cell 7) -> dot product
#   pasti memetakan term yang sama ke index yang sama. Ini PENTING.
#
# OUTPUT:
#   ranking_vsm: DataFrame [Dokumen, Bagian Jurnal, Skor VSM, Ranking VSM]
#   sudah ter-urut.
#
# INTERPRETASI vs Cell 9:
#   Top-2 sama (D4 #1, D25 #2). Tapi D24 melompat dari rank 7 (TF-IDF)
#   ke rank 3 (VSM) -> paragraf pendek-padat-term-query yang dirugikan
#   summation tetapi diuntungkan cosine.
# =====================================================================

def cosine_similarity_manual(vec_a, vec_b):
    pembilang = float(np.dot(vec_a, vec_b))
    penyebut = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return pembilang / penyebut if penyebut != 0 else 0.0

counter_query = Counter(query_terms)
total_query = len(query_terms) if len(query_terms) > 0 else 1
tf_query = {term: counter_query.get(term, 0) / total_query for term in vocab}
vektor_query = np.array([tf_query[term] * idf[term] for term in vocab], dtype=float)

skor_vsm = {}
for dok in nama_dokumen:
    vektor_dok = np.array([tfidf[dok][term] for term in vocab], dtype=float)
    skor_vsm[dok] = cosine_similarity_manual(vektor_query, vektor_dok)

ranking_vsm = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Bagian Jurnal': [label_dokumen[d] for d in nama_dokumen],
    'Skor VSM': [skor_vsm[d] for d in nama_dokumen]
}).sort_values('Skor VSM', ascending=False).reset_index(drop=True)
ranking_vsm['Ranking VSM'] = ranking_vsm.index + 1

display(ranking_vsm)


## Perbandingan Hasil Ranking

In [ ]:
# =====================================================================
# CELL 11 — PERBANDINGAN RANKING TF-IDF vs VSM (LANGKAH 9)
# =====================================================================
# TUJUAN:
#   Menggabungkan dua DataFrame ranking (Cell 9 & 10) menjadi satu
#   tabel side-by-side, sehingga kita bisa MELIHAT karakteristik
#   masing-masing model secara langsung.
#
# CARA:
#   - merge() pada kolom 'Dokumen' (inner join) -> setiap baris
#     menampilkan Skor & Ranking TF-IDF + Skor & Ranking VSM untuk
#     dokumen yang sama.
#   - sort_values('Ranking TF-IDF') -> tampilan diurutkan menurut
#     ranking TF-IDF, jadi pembaca bisa membaca dari "rank 1 sampai
#     rank N" dan langsung membandingkan posisi di kolom VSM.
#
# CARA MEMBACA:
#   - Konsensus kuat: dokumen yang Ranking-TF-IDF dan Ranking-VSM
#     sama (mis. D4 di rank 1, D25 di rank 2 di kedua model) =
#     dokumen yang JELAS relevan dengan query.
#   - Lompatan besar = sinyal bias panjang.
#       D24: TF-IDF rank #7 -> VSM rank #3 (cosine memenangkan
#            paragraf padat term query meski singkat).
#       D18: TF-IDF rank #3 -> VSM rank #6 (paragraf panjang
#            yang term-querynya "encer" ditarik turun cosine).
#   - D6 = 0 di kedua model -> paragraf yang tidak relevan sama
#     sekali, hasil konsisten.
#
# INSIGHT PEDAGOGIS:
#   - TF-IDF summation: menghargai BANYAK match (sensitif panjang).
#   - VSM cosine     : menghargai KEPADATAN match (length-normalized).
# =====================================================================

hasil_perbandingan = (
    ranking_tfidf[['Dokumen', 'Bagian Jurnal', 'Skor TF-IDF', 'Ranking TF-IDF']]
    .merge(ranking_vsm[['Dokumen', 'Skor VSM', 'Ranking VSM']], on='Dokumen', how='inner')
)
hasil_perbandingan = hasil_perbandingan.sort_values('Ranking TF-IDF').reset_index(drop=True)

display(hasil_perbandingan[['Dokumen', 'Bagian Jurnal', 'Skor TF-IDF', 'Ranking TF-IDF', 'Skor VSM', 'Ranking VSM']])


## Summarization

In [ ]:
# =====================================================================
# CELL 12 — EXTRACTIVE SUMMARIZATION (LANGKAH 10)
# =====================================================================
# IDE INTI:
#   Untuk tiap paragraf D1..D26:
#     1) Pecah jadi kalimat-kalimat individual.
#     2) Skor tiap kalimat = rata-rata TF-IDF token-token-nya.
#     3) Pilih 1 kalimat dengan skor tertinggi -> kalimat ringkasan.
#   Gabungkan 26 kalimat terpilih -> ringkasan akhir jurnal.
#
# ---------------------------------------------------------------------
# A) normalisasi_teks(teks)
# ---------------------------------------------------------------------
#   Perbaikan kosmetik sebelum splitting:
#     - Ganti smart quotes "/"/'/' menjadi quote ASCII " dan '
#       (kalau tidak, regex matching jadi tidak konsisten).
#     - Tekan whitespace ganda jadi 1 spasi.
#
# ---------------------------------------------------------------------
# B) pecah_kalimat(teks)  -> SENTENCE SPLITTER ANTI-FALSE-POSITIVE
# ---------------------------------------------------------------------
#   Naive split pada setiap '.' akan rusak karena:
#     - Desimal/jam:     "11.00", "12.00"
#     - Inisial nama:    "A. Tedd", "A.Tedd"
#     - Singkatan umum:  "JL.", "No.", "dkk.", "dll.", "tsb."
#     - Tanggal numerik: "28. November"
#
#   STRATEGI: PLACEHOLDER PROTECTION
#     1) Cari semua titik-bukan-akhir lewat regex `pola_titik_bukan_akhir`,
#        ganti tiap match dengan token unik `__P0__`, `__P1__`, ...
#        Simpan mapping di dict `pelindung`.
#     2) SEKARANG aman split: re.split(r'(?<=[.!?])\s+(?=[A-Z"\d])', teks)
#        -> hanya split saat titik diikuti spasi + huruf besar/kutip/angka.
#        Lookbehind (?<=...) dan lookahead (?=...) memastikan posisi
#        tepat tanpa "memakan" karakternya, jadi titik tetap ada di
#        kalimat sebelumnya dan huruf besar tetap di awal kalimat baru.
#     3) Pulihkan tiap placeholder ke teks asli.
#     4) Buang kalimat < 25 char (noise / artefak split).
#     5) Buang titik trailing (akan dipasang ulang di tahap display).
#
# ---------------------------------------------------------------------
# C) skor_kalimat(kalimat, ...)
# ---------------------------------------------------------------------
#   Rumus:  skor = (Σ tf_lokal(t) * idf_korpus(t)) / jumlah_token
#     - tf_lokal(t) dihitung relatif PANJANG KALIMAT, bukan paragraf.
#     - idf yang dipakai adalah idf KORPUS (Cell 7), karena IDF
#       memang properti tingkat-korpus.
#     - Pembagian akhir dengan jumlah token = RATA-RATA, bukan jumlah
#       -> supaya kalimat panjang tidak otomatis menang. (Trade-off:
#       kalimat satu-kata dengan term langka bisa "menang" semu, tapi
#       umumnya kalimat <25 char sudah dibuang di splitter.)
#
# ---------------------------------------------------------------------
# D) LOOP UTAMA
# ---------------------------------------------------------------------
#   for setiap dokumen:
#     - panggil pecah_kalimat() untuk mendapatkan list kalimat.
#     - skor tiap kalimat.
#     - max() ambil kalimat ber-skor tertinggi -> kalimat_terbaik.
#     - cetak skor semua kalimat (debug-friendly), tandai yang TERPILIH.
#     - simpan ke summary_rows (untuk tabel) & kalimat_summary (untuk
#       paragraf ringkasan akhir).
#
# OUTPUT AKHIR:
#   - df_summary_result : tabel per dokumen
#                          [Dokumen, Bagian Jurnal, Jumlah Kalimat,
#                           Skor Kalimat Terpilih, Kalimat Summary].
#   - ringkasan_akhir   : 26 kalimat terpilih digabung menjadi satu
#                          paragraf ringkasan ekstraktif jurnal.
# =====================================================================

def normalisasi_teks(teks: str) -> str:
    """Bersihkan smart quotes & whitespace ganda agar tampilan kalimat rapi."""
    teks = (teks
            .replace('“', '"').replace('”', '"')
            .replace('‘', "'").replace('’', "'"))
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks


def pecah_kalimat(teks: str) -> list[str]:
    """Pecah teks jadi kalimat dengan regex yang menghormati:
       - desimal/jam (11.00, 12.00)         -> tidak dipecah
       - singkatan 1 huruf (A.Tedd, A. Tedd) -> tidak dipecah
       - singkatan 'JL.', 'No.', 'dkk.'      -> tidak dipecah
       - tanggal: 28 November                -> tidak dipecah
    """
    teks = normalisasi_teks(teks)

    # Lindungi pola titik yang BUKAN akhir kalimat dengan placeholder.
    pelindung = {}
    def lindungi(m):
        token = f'__P{len(pelindung)}__'
        pelindung[token] = m.group(0)
        return token

    pola_titik_bukan_akhir = re.compile(
        r'\b(?:'
          r'\d+\.\d+'                       # 11.00, 12.00
          r'|[A-Z]\.\s?[A-Z][a-z]+'         # A. Tedd / A.Tedd
          r'|JL\.|No\.|dkk\.|dll\.|tsb\.'   # singkatan umum
          r'|\d+\.\s?[A-Z][a-z]+'           # tanggal: 28 November
        r')',
        flags=re.IGNORECASE,
    )
    teks_aman = pola_titik_bukan_akhir.sub(lindungi, teks)

    # Pecah pada titik/!/? yang diikuti spasi + huruf besar / kutip / angka.
    bagian = re.split(r'(?<=[.!?])\s+(?=[A-Z"\d])', teks_aman)

    # Pulihkan placeholder kembali jadi teks asli
    hasil = []
    for kal in bagian:
        for token, asli in pelindung.items():
            kal = kal.replace(token, asli)
        kal = kal.strip()                              # buang whitespace pinggir
        kal = re.sub(r'\s*\.+\s*$', '', kal)            # buang titik akhir (dipasang ulang saat display)
        if len(kal) >= 25:
            hasil.append(kal)
    return hasil


def skor_kalimat(kalimat, tfidf_dok, vocab, idf):
    """Skor kalimat = rata-rata TF-IDF semua token dalam kalimat."""
    tokens = preprocess(kalimat)
    if not tokens:
        return 0.0
    total_skor = 0.0
    for token in tokens:
        if token in idf:
            tf_token = tokens.count(token) / len(tokens)
            total_skor += tf_token * idf[token]
    return total_skor / len(tokens)


print('SUMMARIZATION JURNAL BERBASIS TF-IDF')
print('=' * 70)
print(f'Judul Jurnal : Perkembangan dan Peran OPAC pada Aplikasi CIP (Cerah Informasi Pustaka)')
print(f'Penulis      : Betari Ayu Elsadantia')
print(f'Query        : {query}')
print('=' * 70)

summary_rows = []
kalimat_summary = []

for dok in nama_dokumen:
    teks_asli = dokumen[dok]

    # Pecah dokumen menjadi kalimat dengan splitter yang melindungi singkatan
    kalimat_list = pecah_kalimat(teks_asli)

    if not kalimat_list:
        continue

    # Hitung skor setiap kalimat
    skor_per_kalimat = [(kal, skor_kalimat(kal, tfidf[dok], vocab, idf))
                       for kal in kalimat_list]

    # Ambil kalimat dengan skor tertinggi
    kalimat_terbaik, skor_terbaik = max(skor_per_kalimat, key=lambda x: x[1])

    print(f'\n{dok} — {label_dokumen[dok]}')
    print('-' * 70)
    print('Skor semua kalimat:')
    for i, (kal, skor) in enumerate(skor_per_kalimat, 1):
        tanda = ' ◄ TERPILIH' if kal == kalimat_terbaik else ''
        print(f'  [{i}] (skor: {skor:.5f}){tanda}')
        print(f'       "{kal[:80]}..."' if len(kal) > 80 else f'       "{kal}"')
    print(f'\n  >>> Kalimat Terpilih (skor: {skor_terbaik:.5f}):')
    print(f'      "{kalimat_terbaik}."')

    kalimat_summary.append(kalimat_terbaik + '.')
    summary_rows.append({
        'Dokumen': dok,
        'Bagian Jurnal': label_dokumen[dok],
        'Jumlah Kalimat': len(kalimat_list),
        'Skor Kalimat Terpilih': round(skor_terbaik, 5),
        'Kalimat Summary': kalimat_terbaik + '.'
    })

# Tampilkan tabel summary
print('\n')
print('=' * 70)
print('TABEL HASIL SUMMARIZATION PER DOKUMEN')
print('=' * 70)
df_summary_result = pd.DataFrame(summary_rows)
display(df_summary_result)

# Tampilkan ringkasan akhir
print('\n')
print('=' * 70)
print('RINGKASAN AKHIR JURNAL (EXTRACTIVE SUMMARIZATION)')
print('=' * 70)
ringkasan_akhir = ' '.join(kalimat_summary)
print(ringkasan_akhir)
